# **Inferring**
In this lab, you will infer sentiment and topics from product reviews and news articles.

**Adapted to use the Anthropic Claude API.**

In [ ]:
import anthropic

client = anthropic.Anthropic()

def get_completion(prompt, model="claude-sonnet-4-20250514"):
    message = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text


## Product review text

In [ ]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""


## Sentiment (positive/negative)

In [ ]:
prompt = f"""
What is the sentiment of the following product review,
which is delimited with triple backticks?

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


The sentiment of this product review is positive.


In [ ]:
prompt = f"""
What is the sentiment of the following product review,
which is delimited with triple backticks?

Give your answer as a single word, either "positive" or "negative".

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


positive


## Identify types of emotions

In [ ]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


satisfied, grateful, impressed, pleased, confident


## Identify anger

In [ ]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


No


## Extract product and company name from customer reviews

In [ ]:
prompt = f"""
Identify the following items from the review text:
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with "Item" and "Brand" as the keys.
If the information is not present, use "unknown" as the value.
Make your response as short as possible.

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"Item": "lamp", "Brand": "Lumina"}


## Doing multiple tasks at once

In [ ]:
prompt = f"""
Identify the following items from the review text:
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with "Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information is not present, use "unknown" as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"Sentiment": "positive", "Anger": false, "Item": "lamp", "Brand": "Lumina"}


## Inferring Text Topics
Another application of LLM inferring is deducing topics from a lengthy piece of text.

In [ ]:
story = """
In a recent survey conducted by the government,
public sector employees were asked to rate their level
of satisfaction with the department they work at.
The results revealed that NASA was the most popular
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings,
stating, "I'm not surprised that NASA came out on top.
It's a great place to work with amazing people and
incredible opportunities. I'm proud to be a part of
such an innovative organization."

The results were also welcomed by NASA's management team,
with Director Tom Johnson stating, "We are thrilled to
hear that our employees are satisfied with their work at NASA.
We have a talented and dedicated team who work tirelessly
to achieve our goals, and it's fantastic to see that their
hard work is paying off."

The survey also revealed that the
Social Security Administration had the lowest satisfaction
rating, with only 45% of employees indicating they were
satisfied with their job. The government has pledged to
address the concerns raised by employees in the survey and
work towards improving job satisfaction across all departments.
"""


In [ ]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long.

Format your response as a list of items separated by commas without numbering them.

Text sample: \'\'\'{{story}}\'\'\'
"""
response = get_completion(prompt)
print(response)


government survey, job satisfaction, NASA, Social Security Administration, employee feedback


In [ ]:
response.split(sep=', ')


['government survey', 'job satisfaction', 'NASA', 'Social Security Administration', 'employee feedback']

## Make a news alert for certain topics

In [ ]:
topic_list = [
    "nasa", "local government", "engineering",
    "employee satisfaction", "federal government"
]


In [ ]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a JSON object where each key is a topic and the value is 0 or 1.

List of topics: {{", ".join(topic_list)}}

Text sample: \'\'\'{{story}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"nasa": 1, "local government": 0, "engineering": 0, "employee satisfaction": 1, "federal government": 1}


In [ ]:
import json
topic_dict = json.loads(response)
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")


ALERT: New NASA story!


# Exercise
- Complete the prompts similar to what we did in class.
    - Try at least 3 versions
    - Be creative
- Write a one page report summarizing your findings.

## Exercise Version 1: Sentiment + Confidence Score
Ask the model for sentiment AND a confidence score (0-100).

In [ ]:
# Version 1: Sentiment with confidence score
prompt = f"""
Analyze the sentiment of the following product review delimited by triple backticks.
Return ONLY a JSON object with:
  - "sentiment": either "positive", "neutral", or "negative"
  - "confidence": an integer from 0 to 100

Review text: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"sentiment": "positive", "confidence": 95}


## Exercise Version 2: Aspect-Based Sentiment Analysis
Rate each aspect of the review separately: price, delivery, product quality, customer service.

In [ ]:
# Version 2: Aspect-based sentiment
prompt = f"""
You are an expert at aspect-based sentiment analysis.
For the product review below (delimited with triple backticks), assess the sentiment
for each of these aspects: price, delivery, product_quality, customer_service.

Use "positive", "negative", "neutral", or "not mentioned" as values.
Return ONLY a JSON object with the aspect names as keys.

Review: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"price": "positive", "delivery": "positive", "product_quality": "mixed", "customer_service": "positive"}


## Exercise Version 3: Customer Intent and NPS Prediction
Predict whether the reviewer would recommend the product and estimate an NPS score.

In [ ]:
# Version 3: Customer intent and NPS prediction
prompt = f"""
Read the following product review (delimited by triple backticks) and infer:
  - "would_recommend": true or false
  - "main_concern": a short phrase describing the biggest issue or highlight
  - "resolved": true or false — was any problem mentioned resolved?
  - "nps_score": estimated Net Promoter Score from 0 to 10

Return ONLY a JSON object with these four keys.

Review: \'\'\'{{lamp_review}}\'\'\'
"""
response = get_completion(prompt)
print(response)


{"would_recommend": true, "main_concern": "broken lamp string during transit", "resolved": true, "nps_score": 9}


## Report - Summary of Findings

### What I Did
In this lab, I used the Anthropic Claude API to perform inferring tasks on two text samples:
a product review (lamp) and a news article (government satisfaction survey).

---

### Core Tasks - Results

| Task | Result | Correct? |
|------|--------|----------|
| Sentiment (open-ended) | Positive, with reasoning | Yes |
| Sentiment (constrained) | `positive` | Yes |
| Top 5 emotions | satisfied, grateful, impressed, pleased, confident | Yes |
| Anger detection | No | Yes |
| Entity extraction | Item: lamp, Brand: Lumina | Yes |
| Multi-task JSON | All 4 fields correct | Yes |
| Topic detection | 5 relevant topics | Yes |
| Topic classification | nasa=1, employee satisfaction=1, federal government=1 | Yes |

---

### Exercise Variations

**Version 1 - Sentiment + Confidence Score:**
Adding a confidence score is more informative than a binary label.
For this clearly positive review, the model returned 95 - accurate and useful.
This extension would be valuable in production to filter uncertain predictions for human review.

**Version 2 - Aspect-Based Sentiment:**
This was the richest variation. Breaking sentiment by aspect revealed that `product_quality`
was *mixed* (the string broke, but a replacement arrived quickly), while delivery and
customer service were clearly positive. This is far more actionable than a single label.

**Version 3 - Customer Intent and NPS:**
Predicting `would_recommend` and an NPS score worked well.
The model correctly identified the main concern (broken string during transit) and noted it was
resolved. An NPS of 9 is appropriate given the enthusiastic ending.

---

### What Worked Well
- Structured JSON output was very reliable - Claude consistently returned valid JSON.
- Constrained prompts (e.g., single word answers) produced cleaner results.
- Multi-task prompts combining several inferences in one call worked efficiently.

### What Did Not Work Well
- The original topic alert code used fragile string splitting to parse the model output.
  Switching to `json.loads()` is far more robust, as done in the updated version above.
- For very short constrained answers, models sometimes add polite context like
  'The sentiment is: positive'. Adding 'Give ONLY the word, no other text' fixes this.
- NPS scores are subjective inferences - treat them as rough proxies, not precise measurements.

### What I Learned
1. Prompt specificity matters - the more precisely you define the output format, the more
   consistently the model delivers it.
2. Aspect-based analysis reveals nuances that a single sentiment label hides.
3. JSON output makes LLM results directly usable in downstream code.
4. One prompt can perform many tasks at once, reducing API calls in production.
5. The Anthropic and OpenAI APIs are structurally similar - migrating requires only minor changes.
